*****Automated Quality Control in
Manufacturing*****

In [2]:
import os

data_dir = "/kaggle/input/mobile-screen-data/"


main_directory_contents = [d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))]


subdirectory_contents = {}
for subdir in main_directory_contents:
    subdir_path = os.path.join(data_dir, subdir)
    subdirs = [d for d in os.listdir(subdir_path) if os.path.isdir(os.path.join(subdir_path, d))]
    if subdirs:
        subdirectory_contents[subdir] = subdirs

main_directory_contents, subdirectory_contents

FileNotFoundError: [WinError 3] The system cannot find the path specified: '/kaggle/input/mobile-screen-data/'

Division of clsses

In [ ]:
import os
import pandas as pd
import warnings
warnings.filterwarnings("ignore")


def Create_Directory_DataFrame():
    data = []  
    basedir = '/kaggle/input/mobile-screen-data'

    for Class in os.listdir(basedir):
        for location in os.listdir(os.path.join(basedir, Class)):
            data.append({'Class': Class, 'Location': os.path.join(basedir, Class, location)})

    df = pd.DataFrame(data)
    df = df.sample(frac=1).reset_index(drop=True)  

    return df
import pandas as pd




df = Create_Directory_DataFrame()
print(df.shape)
df.head()

In [ ]:

df['Class'] = df['Class'].replace({'ground_truth_1': 0, 'ground_truth_2': 1,'oil': 2, 'scratch': 3,'stain': 4, 'good': 5})
df.head()

Data preprocessing

In [ ]:
import librosa
import librosa.display
import matplotlib.pyplot as plt
import os
import numpy as np
import cv2
count = 1
f = plt.figure(figsize=(50,13))
for Class in df['Class'].unique():
    seg = df[df['Class']==Class]
    address =  seg.sample().iloc[0]['Location']
    img = cv2.imread(address,0)
    ax = f.add_subplot(2, 5,count)
    ax = plt.imshow(img)
    ax = plt.title(Class,fontsize= 30)
    count = count + 1
plt.show()

In [ ]:
w , h= 32,32
final_class = 6

In [ ]:
from tqdm import tqdm
from sklearn.preprocessing import OneHotEncoder
train_image = []
for location in tqdm(df.iloc[:]['Location']):
    img = cv2.imread(location,0)
    img = cv2.resize(img, (w,h), interpolation = cv2.INTER_AREA)
    img = img.reshape(w,h,1)
    train_image.append(img)
X = np.array(train_image)
y = np.array(df.iloc[:]['Class'])
y = y.reshape(y.shape[0],1)
enc = OneHotEncoder(handle_unknown='ignore')
enc.fit(y)
print(enc.categories_)
y = enc.transform(y).toarray()
print('Data   :   '+str(X.shape))
print('Output :   '+str(y.shape))

Spliting the data

In [ ]:
from sklearn.model_selection import train_test_split
import tensorflow as tf

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=0.2)

In [ ]:
import cv2
import numpy as np


resized_images = []


for image in X_train:
    
    resized = cv2.resize(image, (192, 108))  
    
    
    rgb_image = np.repeat(resized[..., np.newaxis], 3, axis=-1)
    
    resized_images.append(rgb_image)


X_train = np.array(resized_images)


resized_images = []
for image in X_test:
    resized = cv2.resize(image, (192, 108))
    rgb_image = np.repeat(resized[..., np.newaxis], 3, axis=-1)
    resized_images.append(rgb_image)
X_test = np.array(resized_images)


print("Shape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)

In [ ]:
print('Train data    :'+str(X_train.shape))
print('Test data     :'+str(X_test.shape))
print('Train Output  :'+str(y_train.shape))
print('Test Output   :'+str(y_test.shape))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from keras.layers import Flatten, Dense
from keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, load_img
from keras.applications.mobilenet import MobileNet, preprocess_input 
from keras.losses import categorical_crossentropy

Making of the model 

In [ ]:

num_classes = 6


IMAGE_SIZE = [192, 108]

mobilenet = MobileNet( input_shape=(108, 192, 3), include_top= False )

for layer in mobilenet.layers:
  layer.trainable = False


x = Flatten()(mobilenet.output)
x = Dense(units=6 , activation='softmax' )(x)


model = Model(mobilenet.input, x)
model.summary()

In [ ]:
model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)
history = model.fit(X_train, y_train, epochs=100, validation_split=0.3, batch_size=15, verbose=1, shuffle=True)

In [ ]:
import matplotlib.pyplot as plt


plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')


plt.subplot(1, 2, 2)
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.tight_layout()
plt.show()
model.save('Mobile.h5')

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing import image
import numpy as np
import os

# Load the trained model
model = tf.keras.models.load_model("/kaggle/working/best_model.h5")

def predict_image(img_path):
    # Load and preprocess the image
    img = image.load_img(img_path, target_size=(192, 108))
    img_array = image.img_to_array(img)
    img_array = img_array / 255.0  # Normalize
    img_array = np.expand_dims(img_array, axis=0)  # Expand dimensions for model input shape
    
    # Make a prediction
    predictions = model.predict([img_array, img_array, img_array])  # As the model has 3 inputs
    predicted_class = np.argmax(predictions, axis=1)
    
    # Map the predicted class index to the respective class
    class_map = ['good', 'oil', 'scratch', 'stain']
    return class_map[predicted_class[0]]

# Test with a set of provided images
image_directory = "//kaggle/input/mobile-screen-data/oil"
image_files = os.listdir(image_directory)

for img_file in image_files:
    img_path = os.path.join(image_directory, img_file)
    prediction = predict_image(img_path)
    print(f"Image: {img_file}, Predicted Class: {prediction}")

In [ ]:
import streamlit as st
import tensorflow as tf
from tensorflow.keras.preprocessing import image
import numpy as np
import screen  # Import your screen.py module

# Load the trained model
model = tf.keras.models.load_model("/path/to/your/best_model.h5")

def predict_image(img):
    img = img.resize((192, 108))  # Resize image to match model input
    img_array = image.img_to_array(img)
    img_array = img_array / 255.0  # Normalize
    img_array = np.expand_dims(img_array, axis=0)  # Expand dimensions for model input shape

    # Make a prediction
    predictions = model.predict([img_array, img_array, img_array])  # As the model has 3 inputs
    predicted_class = np.argmax(predictions, axis=1)
    
    # Map the predicted class index to the respective class
    class_map = ['good', 'oil', 'scratch', 'stain']
    return class_map[predicted_class[0]]

# Streamlit interface
st.title('Mobile Screen Damage Classifier')

uploaded_file = st.file_uploader("Choose an image...", type="jpg")

if uploaded_file is not None:
    img = image.load_img(uploaded_file)
    st.image(img, caption='Uploaded Image.', use_column_width=True)
    st.write("")
    st.write("Classifying...")

    prediction = predict_image(img)
    st.write(f'Prediction: {prediction}')


In [ ]:
!streamlit run screen.py

In [ ]:
!streamlit run screen.py